# Exercise 9: series

* pandas series vs numpy arrays [explanation](https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html)

### Common series operations
These are the most common series operations we use. Refer to the `pandas` docs for even more!

* Getting dates, hours, minutes from datetime types (`df.datetime_col.dt.date`)
* Parsing strings (`df.string_col.str.split('_')`)

### Common geoseries operations
These are the most common. Refer to the `geopandas` docs for even more!

* `distance` between 2 points or a point to a polygon or line [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.distance.html)
* `intersects`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.intersects.html)
* `within`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.within.html)
* `contains`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html)

In fact, we've often used geoseries methods without even realizing it. Often, we'd create a new column that stores either the line's length or a polygon's area. `gdf.geometry` is a geoseries, and we call methods on that geoseries, and add that as a new column.

For calculations like `length`, `area`, and `distance`, we need to use a projected CRS that has units like meters or feet. We cannot use decimal degrees (do not use WGS 84 / EPSG:3326)! Distance calculations must be done only once the spherical 3D Earth has been converted into a 2D plane.

* `length`: get the length of a line (`gdf.geometry.length`)
* `area`: get the area of a polygon (`gdf.geometry.area`)
* `centroid`: get the centroid of a polygon (`gdf.geometry.centroid`)
* `x`: get the x coordinate of a point (`gdf.geometry.x`)
* `y`: get the y coordinate of a point (`gdf.geometry.y`)

### Arrays
* Occasionally, we may even use arrays, especially when the datasets get even larger but we have simple mathematical calculations
* If we need to apply an exponential decay function to a distance column, we essentially want to multiple `distance` by some number
* Since this exponential decay function is somewhat custom and requires us to write our own formula, we would extract the column as a series (`df.distance`) and multiply each value by some other number.
* Even quicker is to use `numpy` with `distance_array = np.array(df.distance)` and get `exponential_array = distance_array*some_number`

In [1]:
import geopandas as gpd
import intake
import numpy as np
import pandas as pd

catalog = intake.open_catalog(
    "../_shared_utils/shared_utils/shared_data_catalog.yml")

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(
/tmp/ipykernel_478/3598373760.py:1: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  im

If you're asking how far is a transit stop from the interstate, you'd want the distance of every point (every row) compared to an interstate highway geometry.

Let's prep the datasets to use series / geoseries to do this.

In [2]:
stops = catalog.ca_transit_stops.read()[["agency", "stop_id", 
                                         "stop_name", "geometry"]]
highways = catalog.state_highway_network.read()

In [5]:
stops.shape, type(stops)

((112150, 4), geopandas.geodataframe.GeoDataFrame)

In [6]:
stops.sample()

,agency,stop_id,stop_name,geometry
86442,Redding,6024,Shasta College,POINT (-122.31761 40.62512)


In [7]:
highways.shape, type(highways)

((1052, 6), geopandas.geodataframe.GeoDataFrame)

In [8]:
highways.sample()

,Route,County,District,RouteType,Direction,geometry
469,89,MNO,9,State,SB,"LINESTRING (-119.52761 38.64266, -119.52901 38..."


Since we want to know the distance from a stop's point to the interstate generally, we need a dissolve. We don't want to compare the distance against the I-5, the I-10 individually, but to the interstate system as a whole.

In [9]:
interstates = (highways[highways.RouteType=="Interstate"]
               .dissolve()
               .reset_index()
               [["geometry"]]
              )

In [10]:
interstates

,geometry
0,"MULTILINESTRING ((-122.06017 39.01932, -122.06..."


In [11]:
# This is still a gdf, just with 1 column
type(interstates)

geopandas.geodataframe.GeoDataFrame

In [12]:
# Pulling out the individual column, it becomes a series/geoseries.
# It's a geoseries here because we had a gdf. 
# If it was a df, it would be a series.
print(type(stops.geometry))
print(type(interstates.geometry))

<class 'geopandas.geoseries.GeoSeries'>
<class 'geopandas.geoseries.GeoSeries'>


Distance is something you can calculate using `geopandas`.

Specifically, it takes a geoseries on the left, and either a geoseries or a single geometry on the right.

An example of having 2 geoseries would be comparing the distance between 2 points. On the left, it would be a geoseries of the origin points and on the right, destination points.

In [14]:
# We get a warning if we leave it in EPSG:4326!
# stops.geometry.distance(interstates.geometry.iloc[0])

In [15]:
stops_geom = stops.to_crs("EPSG:2229").geometry

In [17]:
type(stops_geom)

geopandas.geoseries.GeoSeries

In [18]:
stops_geom.head()

0    POINT (6524314.055 1857471.449)
1    POINT (6523774.224 1856354.299)
2    POINT (6523682.116 1856247.090)
3    POINT (6522553.347 1855159.390)
4    POINT (6522645.615 1855268.365)
Name: geometry, dtype: geometry

In [16]:
interstates_geom = interstates.to_crs("EPSG:2229").geometry.iloc[0]

In [19]:
distance_series = stops_geom.distance(interstates_geom)

In [20]:
distance_series.head()

0    8934.678853
1    7824.405267
2    7718.241352
3    6643.422637
4    6751.262598
dtype: float64

In [21]:
# Let's make sure that for every stop, a distance is calculated
print(f"# rows in stops: {len(stops_geom)}")
print(f"# rows in stops: {len(distance_series)}")

# rows in stops: 112150
# rows in stops: 112150


In [22]:
# distance is numeric, not a geometry, so we're back to being a series
type(distance_series)

pandas.core.series.Series

What can we do with this? 

We usually add it as a new column. Since we did nothing to shift the index, we can just attach the series back to our gdf.

Getting a distance calculation using geoseries is much quicker than a row-wise lambda function where you calculate the distance.

```
Alternative method that's slower:
      
interstate_geom = interstates.geometry.iloc[0]

stops = stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstate_geom))
)   
```

In [23]:
stops = stops.assign(
    distance_to_interstate = distance_series
)

In [24]:
%%timeit
distance_series = stops_geom.distance(interstates_geom)

15.8 s ± 148 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
"""
%%timeit
stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstates_geom))
)  """

'\n%%timeit\nstops.assign(\n   distance = stops.geometry.apply(\n         lambda x: x.distance(interstates_geom))\n)  '

In [27]:
"""
import dask_geopandas as dg

stops_gddf = dg.from_geopandas(stops, npartitions=2)
stops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry """

'\nimport dask_geopandas as dg\n\nstops_gddf = dg.from_geopandas(stops, npartitions=2)\nstops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry '

In [28]:
 """ %%timeit

distance_series = stops_geom_dg.distance(interstates_geom)"""

' %%timeit\n\ndistance_series = stops_geom_dg.distance(interstates_geom)'

## To Do

* Use the `stop_times` table and `stops` table.
* Calculate the straight line distance between the first and last stop for each trip. Call this column `trip_distance`
* Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate. Call this column `shortest_distance_hwy`.
* For each trip, add these 2 new columns, but use series, geoseries, and/or arrays to assign it.
* Provide a preview of the resulting df (do not export)

In [29]:
GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/compiled_cached_views/"
                )

analysis_date = "2023-01-18"
STOP_TIMES_FILE = f"{GCS_FILE_PATH}st_{analysis_date}.parquet"
STOPS_FILE = f"{GCS_FILE_PATH}stops_{analysis_date}.parquet"
highways = catalog.state_highway_network.read()

In [31]:
stop_times = pd.read_parquet(STOP_TIMES_FILE)

In [35]:
stop_times.shape

(3589931, 9)

In [39]:
stop_times.head(2)

,feed_key,trip_id,stop_id,stop_sequence,timepoint,arrival_sec,departure_sec,arrival_hour,departure_hour
0,48138ae7269d615d5509958097039bf7,t287-b194-sl4_merged_3564,1140,11,NaN,25047,25047,6,6
1,48138ae7269d615d5509958097039bf7,t708-b12D-sl4_merged_4213,1161,25,NaN,66583,66583,18,18


#### Prep Highways -> Interstate Only

In [45]:
interstate = (highways[highways.RouteType=="Interstate"]).reset_index(drop = True)

In [47]:
interstate.shape, type(interstate)

((128, 6), geopandas.geodataframe.GeoDataFrame)

In [51]:
interstate_dissolved = interstate[['Route','County','geometry']].dissolve(by = ['Route','County']).reset_index()

In [53]:
interstate_dissolved.shape

(64, 3)

In [55]:
# interstate_dissolved.plot()

In [52]:
interstate_dissolved.sample()

,Route,County,geometry
13,5,STA,"MULTILINESTRING ((-121.08870 37.24607, -121.08..."


#### Prep Stops

In [32]:
stops = gpd.read_parquet(STOPS_FILE)

In [38]:
stops.shape

(84688, 16)

In [43]:
stops_subset = stops[['feed_key', 'stop_id', 'geometry']]

In [44]:
stops_subset.head(2)

,feed_key,stop_id,stop_key,stop_name,geometry
0,6adf6cd9b6d24ab4ee8ee220e3697a73,15193,d4eb0920e7e256606df449c31b3c3e6a,Vanowen / Encino,POINT (136891.364 -423571.990)
1,6adf6cd9b6d24ab4ee8ee220e3697a73,14025,038cca58ef5f071ff5c94b8213989f87,Vermont / 110th,POINT (157907.732 -451883.320)


#### Calculate the straight line distance between the first and last stop for each trip. Call this column trip_distance

In [57]:
stop_times_subset = stop_times[['feed_key','trip_id','stop_id','stop_sequence']]

In [71]:
stop_times_sorted = stop_times_subset.sort_values(by = ['feed_key','trip_id','stop_sequence']).reset_index(drop = True)

In [74]:
# stop_times_sorted.loc[stop_times_sorted.trip_id == "1080037"]

In [73]:
stops_m1 = pd.merge(stops_subset, stop_times_sorted, how = 'inner', on = ['feed_key', 'stop_id'])

In [75]:
len(stops_m1), len(stop_times_sorted), len(stops_subset)

(3589718, 3589931, 84688)

In [80]:
len(stops_m1.drop_duplicates(subset = ['feed_key','stop_id','stop_key','stop_sequence']))

146550

In [81]:
stops_m2 = stops_m1.drop_duplicates(subset = ['feed_key','stop_id','stop_key','stop_sequence']).reset_index(drop = True)

In [83]:
stops_m2 = stops_m2.sort_values(by = ['feed_key','trip_id','stop_sequence']).reset_index(drop = True)

In [94]:
first_stop = stops_m2.drop_duplicates(subset = ['feed_key','trip_id',]).reset_index(drop = True)

In [96]:
# first_stop.head(20)